## This notebook deals with explaining Merge Operation in PySpark in Databricks

In [0]:
from pyspark.sql import Row

# Sample Employee info
data_employee = [
    Row(employee_id=1, name='Alice', city='New York'),
    Row(employee_id=2, name='Bob', city='Los Angeles'),
    Row(employee_id=3, name='Charlie', city='Chicago'),
    Row(employee_id=4, name='David', city='Houston'),
    Row(employee_id=5, name='Eva', city='Phoenix')
]

df_employee = spark.createDataFrame(data_employee)

# Employee info
df_employee.createOrReplaceTempView('employee')

display(df_employee)

In [0]:
%sql
create or replace table test.testing.employee_target_table
as 
select 
    *,
    current_timestamp() as insert_timestamp,
    current_time() as update_timestamp
from employee

In [0]:
%sql
select * from test.testing.employee_target_table

In [0]:
# DBTITLE 1,Create New Dataframe with source data
from pyspark.sql import Row

# Sample Employee info
data_employee = [
    Row(employee_id=1, name='Alice', city='Phoenix'),
    Row(employee_id=2, name='Bob', city='Houston'),
    Row(employee_id=6, name='Alfred', city='Phoenix'),
    Row(employee_id=7, name='Kate', city='Los Angeles')
]

df_employee = spark.createDataFrame(data_employee)

# Employee info
df_employee.createOrReplaceTempView('employee')

display(df_employee)


In [0]:
%sql
merge into test.testing.employee_target_table target
using employee source
on target.employee_id = source.employee_id

when matched then
    update set      
        target.name=source.name,
        target.city=source.city,
        target.update_timestamp=current_timestamp()
        
when not matched then 
    insert (employee_id, name, city, insert_timestamp, update_timestamp)
    values (source.employee_id, source.name, source.city, current_timestamp(), current_timestamp())

In [0]:
%sql
select * from test.testing.employee_target_table

In [0]:
%sql
merge into test.testing.employee_target_table target
using employee source
on target.employee_id = source.employee_id
-- values match in both target and source
when matched then
    update set      
        target.name=source.name,
        target.city=source.city,
        target.update_timestamp=current_timestamp()
-- basically values in target not in source        
when not matched then 
    insert (employee_id, name, city, insert_timestamp, update_timestamp)
    values (source.employee_id, source.name, source.city, current_timestamp(), current_timestamp())
-- basically values in source not in target
when not matched by source then 
    delete

In [0]:
%sql
select * from test.testing.employee_target_table

In [0]:
df_employee.display()

In [0]:
from pyspark.sql import functions as F
df_employee.withColumn('insert_timestamp', F.current_timestamp()).withColumn('update_timestamp', F.current_timestamp()).createOrReplaceTempView('employee')

In [0]:
%sql
-- this insert * and update * can be done when soruce and target have same columns only
merge into test.testing.employee_target_table target
using employee source
on target.employee_id = source.employee_id
-- values match in both target and source
when matched then
    update set *
-- basically values in target not in source        
when not matched then 
    insert *
-- basically values in source not in target
when not matched by source then 
    delete